# UdaPlay AI Research Agent

## Part 2 - Agent Implementation

This notebook implements the agent layer for UdaPlay, an AI research assistant for the video game industry.

UdaPlay combines local retrieval with web search to answer questions about games, publishers, developers, platforms, release dates, and current industry activity.

The agent will:

1. Retrieve relevant information from the local ChromaDB knowledge base
2. Evaluate whether the retrieved information is sufficient and reliable
3. Fall back to web search when local retrieval is insufficient
4. Maintain workflow and conversation state
5. Return clear, structured answers with source information

### Setup

In [14]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [15]:
# Import the necessary libraries
import os
import json

from dotenv import load_dotenv
from tavily import TavilyClient

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, ToolMessage, AIMessage
from lib.tooling import tool

In [16]:
# Load environment variables
load_dotenv()

True

### Tools

UdaPlay uses three tools:
- `retrieve_game`: semantic search over the local vector database
- `evaluate_retrieval`: evaluates whether the retrieved local information is sufficient
- `game_web_search`: searches the web when local retrieval is insufficient

#### Retrieve Game Tool

In [17]:
# Create retrieve_game tool
import chromadb
from chromadb.utils import embedding_functions

embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("CHROMA_OPENAI_API_KEY"),
    api_base=os.getenv("OPENAI_BASE_URL"),
    model_name="text-embedding-3-small"
)

chroma_client = chromadb.PersistentClient(path="chromadb")

collection = chroma_client.get_collection(
    name="udaplay",
    embedding_function=embedding_fn
)

@tool
def retrieve_game(query: str) -> list:
    """
    Semantic search: Finds the most relevant results in the vector database.

    Args:
        query: A question about the video game industry.

    Returns:
        A list of matching game records.
    """
    results = collection.query(
        query_texts=[query],
        n_results=3
    )

    games = []

    for metadata in results["metadatas"][0]:
        games.append(metadata)

    return games

#### Evaluate Retrieval Tool

In [18]:
# Create evaluate_retrieval tool
from pydantic import BaseModel, Field
from lib.parsers import PydanticOutputParser


class EvaluationReport(BaseModel):
    useful: bool = Field(
        description="Whether the retrieved documents are sufficient to answer the question"
    )
    description: str = Field(
        description="Explanation of why the retrieved documents are or are not useful"
    )

@tool
def evaluate_retrieval(question: str, retrieved_docs: list) -> dict:
    """
    Based on the user's question and retrieved documents,
    evaluates whether the documents are sufficient to answer the question.

    Args:
        question: The original user question.
        retrieved_docs: Documents retrieved from the vector database.

    Returns:
        An evaluation containing:
        - useful: whether the documents can answer the question
        - description: explanation of the evaluation
    """

    llm_judge = LLM(model="gpt-4o-mini")

    prompt = f"""
    Your task is to evaluate whether the retrieved documents contain
    enough information to accurately answer the user's question.

    User question:
    {question}

    Retrieved documents:
    {json.dumps(retrieved_docs, indent=2)}

    Determine whether the documents are useful enough to answer the question.
    Give a clear explanation for your decision.
    """

    response = llm_judge.invoke(
        input=prompt,
        response_format=EvaluationReport
    )

    parser = PydanticOutputParser(model_class=EvaluationReport)
    evaluation = parser.parse(response)

    return evaluation.model_dump()

#### Game Web Search Tool

In [19]:
# Create game_web_search tool
tavily_client = TavilyClient(
    api_key=os.getenv("TAVILY_API_KEY")
)


@tool
def game_web_search(question: str) -> list:
    """
    Web search: Finds relevant information about the video game industry
    when the local vector database does not contain enough information.

    Args:
        question: A question about the video game industry.

    Returns:
        A list of web search results containing the title, URL, and content.
    """

    response = tavily_client.search(
        query=question,
        search_depth="advanced",
        max_results=5
    )

    results = []

    for result in response.get("results", []):
        results.append({
            "title": result.get("title"),
            "url": result.get("url"),
            "content": result.get("content")
        })

    return results

### Agent

In [20]:
# Create the Agent abstraction using StateMachine
udaplay_agent = Agent(
    model_name="gpt-4o-mini",
    temperature=0.2,
    instructions="""
You are UdaPlay, an AI research assistant for the video game industry.

Your job is to answer questions about video games, including:
- game titles
- developers and publishers
- release dates
- platforms
- genres
- descriptions
- current company or game activity

Follow this workflow:

1. Always begin by calling retrieve_game with the user's question.
2. After retrieving local results, call evaluate_retrieval using:
   - the original user question
   - the retrieved documents
3. If evaluate_retrieval returns useful=True:
   - answer using the retrieved information
   - do not perform a web search unless the user explicitly asks for current information
4. If evaluate_retrieval returns useful=False, or the retrieved information is incomplete:
   - call game_web_search using the user's question
5. For questions about current or recent activity, prefer web search if the local dataset does not contain current information.
6. Do not invent facts. Base answers only on retrieved documents or web search results.
7. Give a concise, readable final answer.
8. Clearly indicate whether the answer came from the local game database, web search, or both.
""",
    tools=[
        retrieve_game,
        evaluate_retrieval,
        game_web_search
    ]
)

In [21]:
questions = [
    "When were Pokémon Gold and Silver released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for PlayStation 5?"
]

session_id = "udaplay_demo"

for i, question in enumerate(questions, start=1):

    run = udaplay_agent.invoke(
        question,
        session_id=session_id
    )

    final_state = run.get_final_state()
    messages = final_state["messages"]

    # Find the messages for the current question
    start_index = 0

    for index in range(len(messages) - 1, -1, -1):
        if (
            isinstance(messages[index], UserMessage)
            and messages[index].content == question
        ):
            start_index = index
            break

    current_messages = messages[start_index:]

    print("\n" + "=" * 80)
    print(f"Question {i}: {question}")
    print("=" * 80)

    print("\nAgent Workflow / Tool Trace:")

    for message in current_messages:

        if isinstance(message, AIMessage) and message.tool_calls:
            for tool_call in message.tool_calls:
                print(f"\nTool Called: {tool_call.function.name}")
                print(f"Arguments: {tool_call.function.arguments}")

        elif isinstance(message, ToolMessage):
            print(f"\nTool Result ({message.name}):")
            print(message.content)

    final_answer = next(
        message.content
        for message in reversed(current_messages)
        if isinstance(message, AIMessage)
        and message.content
        and not message.tool_calls
    )

    print("\nFinal Answer:")
    print(final_answer)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

Question 1: When were Pokémon Gold and Silver released?

Agent Workflow / Tool Trace:

Tool Called: retrieve_game
Arguments: {"query":"Pokémon Gold and Silver release date"}

Tool Result (retrieve_game):
"[{'Description': 'Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.', 'Name': 'Pokémon Gold and Silver', 'Publisher': 'Nintendo', 'Platform': 'Game Boy Color', 'Genre': 'Role-playing', 'YearOfRelease': 1999}, {'YearOfRelease': 2002, 'Publisher': 'Nintendo', 'Name': 'Pokémon Ruby and Sapphire', 'Platform': 'Game Boy Advance', 'Description': 'Third-generation Pokémon games set in the Hoenn region, featuri